In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
%matplotlib widget
#from torch.optim.lr_scheduler import ReduceLROnPlateau
#import torch
#import torch.optim as optim
#import torch.nn as nn
#from core.benchmarks import *
#from core.CardiacCTdataset import DataLoaderFactory
#from core.CNNmodel import *
#from core.benchmarks import *
import pandas as pd
from core.CVsplits import *
import logging
from core.Log import *
import json
OUTER_FOLDS = 4; INNER_FOLDS = 3

#logging.shutdown()
#setup_logger('INNER_train')
# ['INNER_train', 'OUTER_train', 'OUTER_evaluate', 'Close']
#log = logging.getLogger('INNER_train')


In [ ]:
# Create Holdout dataset for FINAL Model Evaluation
from sklearn.model_selection import train_test_split
from core.Log import load_dataset_info, save_dataset_info


main_dataset = load_dataset_info(file="data/data_info.json")
labels = [lbl['label'] for lbl in main_dataset]
main_training, final_test = train_test_split(main_dataset,
											 test_size=33,     # 33 for 4 OUTER folds
											 stratify=labels,
											 random_state=42)
print(len(final_test))

for sample in main_dataset:
	if sample in final_test: sample['pool'] = 'holdout'
	else: sample['pool'] = 'main'
#save_dataset_info(main_dataset, file="data/data_info.json")


In [ ]:
pool = [i['pool'] for i in main_dataset]
print(f"Total samples: {len(main_dataset)}, Normal cases: {labels.count(0)}, Takotsubo Cases: {labels.count(1)}")
print(f"Main cases: {pool.count('main')}, Holdout Cases: {pool.count('holdout')} ")
pools = list(set(pool))
for p in pools:
	lbls_in_pool = [d['label'] for d in main_dataset if d['pool'] == p]
	print(f"Pool: {p}, Total samples: {len(lbls_in_pool)}, Normal cases: {lbls_in_pool.count(0)}, Takotsubo Cases: {lbls_in_pool.count(1)}")


In [ ]:
from core.CVsplits import create_folds_stats

create_folds_stats(main_dataset, OUTER_K=4, INNER_K=3)


In [ ]:
import itertools
import json
## 1. Define all model-specific hyperparameter sweeps in one dictionary
model_configs = {
	"MultiViewCNN": {
		"LR_SWEEP": [2e-4, 5e-4, 8e-4],
		"DR_SWEEP": [0.2, 0.3, 0.4],
		"WD_SWEEP": [1e-6, 1e-4]
	},
	#"Axial":    {"LR_SWEEP": [2e-4, 5e-4, 8e-4], "WD_SWEEP": [1e-6, 1e-4], "DR_SWEEP": [0.2, 0.3, 0.4]},
	#"Coronal":  {"LR_SWEEP": [2e-4, 5e-4, 8e-4], "WD_SWEEP": [1e-6, 1e-4], "DR_SWEEP": [0.2, 0.3, 0.4]},
	#"Sagittal": {"LR_SWEEP": [2e-4, 5e-4, 8e-4], "WD_SWEEP": [1e-6, 1e-4], "DR_SWEEP": [0.2, 0.3, 0.4]},
}

# 2. Define global parameters that are the same for all models
GLOBAL_PARAMS = {
	"P": 4,
	"Epochs": 30,
}

INNER_CV_parameters = []
ID = 1

# Iterate through each model and its specific configuration
for model_name, config in model_configs.items():

	# Generate all unique combinations of the model's hyperparameters
	# e.g., for MultiViewCNN, this will create (1e-3, 0.3, 1e-4), (1e-3, 0.4, 1e-4), etc.
	hp_combinations = list(itertools.product(
		config['LR_SWEEP'],
		config['DR_SWEEP'],
		config['WD_SWEEP']
	))

	# Loop through outer and inner folds
	for outer_fold_idx in range(1, 5):
		for inner_fold_idx in range(1, 4):

			# Loop through each hyperparameter combination for this model
			for i, (lr, dr, wd) in enumerate(hp_combinations):
				item = {
					"Model": model_name,
					'OUTER_FOLD': outer_fold_idx,
					'INNER_FOLD': inner_fold_idx,
					"HPset": i+1,
					"LR": lr,
					"WD": wd,
					"DR": dr,
					"P": GLOBAL_PARAMS['P'],
					"Epochs": GLOBAL_PARAMS['Epochs'],
					"trained": False
				}
				INNER_CV_parameters.append(item)
				ID += 1

print(f"Total combinations generated: {len(INNER_CV_parameters)}")

with open("NCV_4_3_folds/INNER_experiments.json", "w") as f:
	json.dump(INNER_CV_parameters, f, indent=2)



In [2]:
INNER_CV_parameters = load_from_json("NCV_4_3_folds/INNER_experiments.json")
df = pd.DataFrame(INNER_CV_parameters)
df.to_csv("NCV/INNER_experiments.csv", index=False)
df


Loaded NCV_4_3_folds/INNER_experiments.json.


,Model,OUTER_FOLD,INNER_FOLD,HPset,LR,WD,DR,P,Epochs,trained
0,MultiViewCNN,1,1,1,0.0002,0.000001,0.2,4,30,True
1,MultiViewCNN,1,1,2,0.0002,0.000100,0.2,4,30,True
2,MultiViewCNN,1,1,3,0.0002,0.000001,0.3,4,30,True
3,MultiViewCNN,1,1,4,0.0002,0.000100,0.3,4,30,True
4,MultiViewCNN,1,1,5,0.0002,0.000001,0.4,4,30,True
...,...,...,...,...,...,...,...,...,...,...
211,MultiViewCNN,4,3,14,0.0008,0.000100,0.2,4,30,False
212,MultiViewCNN,4,3,15,0.0008,0.000001,0.3,4,30,False
213,MultiViewCNN,4,3,16,0.0008,0.000100,0.3,4,30,False
214,MultiViewCNN,4,3,17,0.0008,0.000001,0.4,4,30,False


In [2]:
INNER_CV_parameters = load_from_json("NCV_4_3_folds/INNER_experiments.json")
def is_completed(exp):
    return (
        exp.get("Model") == "MultiViewCNN"
        and exp.get("OUTER_FOLD") == 1
        and exp.get("INNER_FOLD") == 2
        and exp.get("HPset") in {8}
    )

updated = 0
for exp in INNER_CV_parameters:
    if is_completed(exp):
        exp["trained"] = True          # normalize to lowercase
        updated += 1
print(f"Marked {updated} experiments as trained=True.")

with open("NCV_4_3_folds/INNER_experiments.json", "w") as f:
    json.dump(INNER_CV_parameters, f, indent=2)
print("INNER_experiments.json updated.")


Loaded NCV_4_3_folds/INNER_experiments.json.
Marked 1 experiments as trained=True.
INNER_experiments.json updated.


In [3]:

INNER_CV_parameters = load_from_json("NCV_4_3_folds/INNER_experiments.json")
filtered = [
	exp for exp in INNER_CV_parameters
	if exp["Model"] == "MultiViewCNN"
	and exp["OUTER_FOLD"] == 1       # 2, 3, 4
	#and exp["INNER_FOLD"] == 1       # 2, 3, 4
	#and exp["HPset"] == 1
	and exp["trained"] == False
]
f"experiments: {len(filtered)}"


Loaded NCV_4_3_folds/INNER_experiments.json.


'experiments: 29'

In [4]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import roc_auc_score, roc_curve, f1_score

def append_experiment_results(item, path="NCV/INNER_summary.jsonl"):
	with open(path, "a") as f:  # append mode
		f.write(json.dumps(item) + "\n")



def train_INNER_model(model, train_loader, val_loader, experiment):
	log = logging.getLogger('INNER_train')
	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	model.to(device)
	epochs = experiment['Epochs']
	model_name  = experiment['Model']
	out=experiment['OUTER_FOLD']
	inn=experiment['INNER_FOLD']
	HP = experiment['HPset']

	LR = experiment['LR']
	WD = experiment['WD']
	P = experiment['P']
	TH = 0.5
	rel_thresh = 5e-3 if np.isclose(LR, 5e-4) else 3e-3
	sch_patience, sch_cooldown = 2, 1

	ES_PATIENCE = max(P, sch_patience + sch_cooldown + 2)  # ≥ 8
	alpha_ema = 0.30  # smoothing for EMA of val loss

	optimizer = optim.Adam(model.parameters(), lr= LR, weight_decay=WD)
	scheduler = ReduceLROnPlateau(optimizer, mode='min',
								  patience=sch_patience, factor=0.5,
								  threshold=rel_thresh, threshold_mode='rel',
								  cooldown=sch_cooldown, min_lr=1e-6)
	criterion = nn.BCEWithLogitsLoss()

	# --- best trackers ---
	best_val_loss = np.inf
	best_epoch    = -1
	best_lr_at_best = LR
	auc_at_best   = np.nan

	# --- EMA & patience ---
	ema_val = None
	best_ema = np.inf
	no_improve = 0

	val_N=len(val_loader.dataset)
	train_N=len(train_loader.dataset)
	#log.info("Model;ExpID;OuterFold;HPset;Epoch;TrainLoss;TrainAcc;ValLoss;ValAcc;AUC;Brier;EMA_ValLoss;LR;NoImprove;LrDrop;EsTriggered;BestValLoss;BestEpoch;WallTimeSec\n")

	print(f" train_N: {train_N}, val_N: {val_N}		↳ Training model... ")
	for epoch in range(epochs):
		model.train()
		running_loss = 0.0
		y_true, y_pred = [], []
		for batch in train_loader:
			axi = batch["axial_image"].to(device)
			cor = batch["coronal_image"].to(device)
			sag = batch["sagittal_image"].to(device)
			met = batch["meta"].to(device)
			lbl = batch["label"].to(device).unsqueeze(1)

			optimizer.zero_grad(set_to_none=True)

			logits = model(axi, cor, sag, met)
			T_loss = criterion(logits, lbl)

			T_loss.backward()
			optimizer.step()

			running_loss += T_loss.item() * lbl.size(0)
			with torch.no_grad():
				probs = torch.sigmoid(logits)
				preds = (probs > TH).long()
				y_true.append(lbl.detach().cpu().numpy())
				y_pred.append(preds.detach().cpu().numpy())

		TrainLoss = running_loss / max(1, train_N)
		y_true = np.concatenate(y_true).reshape(-1)
		y_pred = np.concatenate(y_pred).reshape(-1)
		TrainAcc  = (y_true == y_pred).mean()

		model.eval()
		running_loss = 0.0
		y_true, y_prob, y_pred = [], [], []

		with torch.no_grad():
			for batch in val_loader:
				axi = batch["axial_image"].to(device)
				cor = batch["coronal_image"].to(device)
				sag = batch["sagittal_image"].to(device)
				met = batch["meta"].to(device)
				lbl = batch["label"].to(device).unsqueeze(1)

				logits = model(axi, cor, sag, met)
				V_loss = criterion(logits, lbl)
				running_loss += V_loss.item() * lbl.size(0)

				probs = torch.sigmoid(logits)
				preds = (probs > TH).long()

				y_true.append(lbl.detach().cpu().numpy())
				y_prob.append(probs.detach().cpu().numpy())
				y_pred.append(preds.detach().cpu().numpy())

		ValLoss = running_loss / max(1, val_N)

		y_true  = np.concatenate(y_true).reshape(-1)
		y_prob  = np.concatenate(y_prob).reshape(-1)
		y_pred  = np.concatenate(y_pred).reshape(-1)

		ValAcc  = (y_true == y_pred).mean()
		AUC = roc_auc_score(y_true, y_prob)

		# -------------- EMA + scheduler --------------
		ema_val = ValLoss if ema_val is None else alpha_ema*ValLoss + (1 - alpha_ema)*ema_val
		prev_lr = optimizer.param_groups[0]['lr']
		scheduler.step(ema_val)  # schedule on EMA, not raw ValLoss
		new_lr = optimizer.param_groups[0]['lr']
		lr_drop = int(new_lr < prev_lr)

		# -------------- early stopping test -----------
		improved = ema_val < best_ema * (1 - rel_thresh)
		if improved:
			best_ema = ema_val
			best_val_loss = ValLoss
			best_epoch = epoch
			best_lr_at_best = new_lr
			auc_at_best = AUC
			no_improve = 0
		else:
			no_improve += 1

		es_triggered = int(no_improve >= ES_PATIENCE)
		line=f"{model_name};{out};{inn};{HP};{epoch};{TrainLoss};{TrainAcc};{ValLoss};{ValAcc};{AUC};{ema_val};{new_lr};{no_improve};{lr_drop};{es_triggered};{best_val_loss};{best_epoch}"
		log.info(line)
		if es_triggered: break

# ----- final return (best state + summary for outer fold) -----
	summary = {
        "Model": model_name,
        "OuterFold": out,
        "Innerfold": inn,
        "LR": LR,
        "WD": WD,
        "BestEpoch": best_epoch,
        "BestValLoss": float(best_val_loss),
        "AUC_at_Best": float(auc_at_best) if auc_at_best is not None else np.nan,
        "LR_at_Best": float(best_lr_at_best),
        "ES_Patience_Used": ES_PATIENCE,
        "RelThresh": rel_thresh,
        "EMA_alpha": alpha_ema,
    }
	return summary



In [5]:
main_dataset = load_dataset_info(file="data/data_info.json")
DL = DataLoaderFactory(main_dataset)


log.info("Model;ExpID;OuterFold;InnerFold;HPset;Epoch;TrainLoss;TrainAcc;ValLoss;ValAcc;AUC;EMA_ValLoss;LR;NoImprove;LrDrop;EsTriggered;BestValLoss;BestEpoch")


In [ ]:

INNER_CV_parameters = load_from_json("NCV_4_3_folds/INNER_experiments.json")
filtered = [
	exp for exp in INNER_CV_parameters
	if exp["Model"] == "MultiViewCNN"
	and exp["OUTER_FOLD"] == 1       # 2, 3, 4
	#and exp["INNER_FOLD"] == 1       # 2, 3,
	#and exp["HPset"] == 1
	and exp["trained"] == False
]
print(f"experiments: {len(filtered)}")

for experiment in filtered:
	print(experiment)
	OUT = experiment['OUTER_FOLD']
	INN = experiment['INNER_FOLD']
	train_loader, val_loader = DL.create_inner_loaders(OUT-1, INN-1)
	DR = experiment['DR']
	model = MultiViewCNN(DR)
	results = train_INNER_model(model, train_loader, val_loader, experiment)
	append_experiment_results(results)
	experiment["trained"] = True

save_to_json(INNER_CV_parameters, "NCV_4_3_folds/INNER_experiments.json")




Loaded NCV_4_3_folds/INNER_experiments.json.
experiments: 29
{'Model': 'MultiViewCNN', 'OUTER_FOLD': 1, 'INNER_FOLD': 2, 'HPset': 8, 'LR': 0.0005, 'WD': 0.0001, 'DR': 0.2, 'P': 4, 'Epochs': 30, 'trained': False}
 train_N: 62, val_N: 31		↳ Training model... 
{'Model': 'MultiViewCNN', 'OUTER_FOLD': 1, 'INNER_FOLD': 2, 'HPset': 9, 'LR': 0.0005, 'WD': 1e-06, 'DR': 0.3, 'P': 4, 'Epochs': 30, 'trained': False}
 train_N: 62, val_N: 31		↳ Training model... 


In [ ]:
def load_experiments_results(path="NCV_5_3_folds/INNER_results.jsonl"):
	experiments = []
	with open(path, "r") as f:
		for line in f:
			if line.strip():
				experiments.append(json.loads(line))
	return experiments


experiments_results = load_experiments_results(path="NCV_5_3_folds/INNER_experiments.json")
experiments_results


In [ ]:
INNER_CV_parameters = load_from_json("NCV_5_3_folds/INNER_experiments.json")
experiments_results = load_experiments_results(path="NCV_5_3_folds/INNER_results.jsonl")
print(f"Total experiments: {len(INNER_CV_parameters)}, Total results: {len(experiments_results)}")



In [ ]:
def summarize_inner_cv(df, metric='best_val_auc'):
	"""
	Summarize inner-CV results per (Model, OUTER_FOLD, HPsetID) and select the best HPset.
	Returns (summary_df, winners_df).
	"""

	# keep only rows that have the metric
	dfm = df.dropna(subset=[metric]).copy()

	# aggregate per HPset across INNER_FOLDs
	summary = (
		dfm
		.groupby(['Model', 'OUTER_FOLD', 'HPsetID'], dropna=False)
		.agg(
			mean_metric=(metric, 'mean'),
			std_metric=(metric, 'std'),
			n_inner=('INNER_FOLD', 'nunique'),
			mean_val_loss=('best_val_loss', 'mean'),
			std_val_loss=('best_val_loss', 'std'),
			mean_threshold=('best_threshold', 'mean'),
			std_threshold=('best_threshold', 'std'),
		)
		.reset_index()
		.sort_values(['Model', 'OUTER_FOLD', 'HPsetID'])
	)
	return summary

INNER_CV_parameters = load_from_json("NCV_5_3_folds/INNER_experiments.json")
INNER_CV_parameters_df = pd.DataFrame(INNER_CV_parameters)
summary = summarize_inner_cv(INNER_CV_parameters_df, metric='best_val_auc')




In [ ]:
import pandas as pd
from collections import Counter

# --- 1) winner selection from the summary produced by summarize_inner_cv ---
def pick_winners(summary: pd.DataFrame, higher_is_better: bool = True) -> pd.DataFrame:
	if summary.empty:
		return summary.copy()

	ascending_metric = not higher_is_better
	# sort by selection keys, then take head(1) per (Model, OUTER_FOLD)
	sort_keys = ['Model', 'OUTER_FOLD', 'mean_metric']
	sort_asc  = [True,     True,           ascending_metric]

	if 'mean_val_loss' in summary.columns:
		sort_keys += ['mean_val_loss']
		sort_asc  += [True]  # lower is better

	if 'std_metric' in summary.columns:
		sort_keys += ['std_metric']
		sort_asc  += [True]  # lower is better

	sort_keys += ['HPsetID']
	sort_asc  += [True]

	ranked = summary.sort_values(sort_keys, ascending=sort_asc)
	winners = (
		ranked
		.groupby(['Model', 'OUTER_FOLD'], as_index=False, sort=False)
		.head(1)
		.reset_index(drop=True)
	)
	return winners


# --- 2) helpers to extract a single hypers dict per (Model, OUTER_FOLD, HPsetID) ---
def _coalesce_mode(values):
	"""Return the most common non-null value, else None."""
	vals = [v for v in values if pd.notna(v)]
	if not vals:
		return None
	return Counter(vals).most_common(1)[0][0]

def _extract_hypers_for_hpset(inner_df: pd.DataFrame, model: str, outer_fold: int, hpset_id) -> dict:
	"""
	Find all INNER experiments matching (Model, OUTER_FOLD, HPsetID) or (hypers['HPset'] == HPsetID).
	From those rows, coalesce fields LR/WD/DR/P/Epochs into a single hypers dict.
	"""
	dfm = inner_df.copy()
	# Ensure HPsetID column exists even if it was only inside hypers
	if 'HPsetID' not in dfm.columns:
		if 'hypers' in dfm.columns:
			dfm['HPsetID'] = dfm['hypers'].apply(lambda h: (h or {}).get('HPset') if isinstance(h, dict) else pd.NA)
		else:
			dfm['HPsetID'] = pd.NA

	cand = dfm[(dfm['Model'] == model) & (dfm['OUTER_FOLD'] == outer_fold)]
	# match either by explicit HPsetID or by hypers['HPset']
	cand = cand[(cand['HPsetID'] == hpset_id) | (
		cand.get('hypers') is not None and
		cand['hypers'].apply(lambda h: isinstance(h, dict) and h.get('HPset') == hpset_id)
	)]

	# pull fields from 'hypers' dict; coalesce if multiple rows
	def pull(field, default=None):
		vals = []
		for _, r in cand.iterrows():
			h = r.get('hypers', {}) if isinstance(r.get('hypers'), dict) else {}
			vals.append(h.get(field, default))
		return _coalesce_mode(vals)

	hypers = {
		'HPset':   hpset_id,
		'LR':      pull('LR'),
		'WD':      pull('WD'),
		'DR':      pull('DR'),
		'TH':      None,  # set later from summary mean_threshold
		'P':       pull('P', 5),
		'Epochs':  50,
	}
	return hypers


# --- 3) main constructor: OUTER experiments list ---
def build_outer_experiments(summary_df: pd.DataFrame,
							inner_params_records: list,
							higher_is_better: bool = True,
							threshold_source: str = 'mean_threshold',
							start_exp_id: int = 1) -> list:
	"""
	From the summary (one row per Model×OUTER_FOLD×HPsetID with mean_*) and the raw INNER records,
	choose the optimal HPset per (Model, OUTER_FOLD) and create a list of OUTER experiment dicts.

	threshold_source: which column in summary to use for TH (usually 'mean_threshold').
	"""
	inner_df = pd.DataFrame(inner_params_records)
	winners = pick_winners(summary_df, higher_is_better=higher_is_better)

	experiments = []
	exp_id = start_exp_id

	for _, w in winners.iterrows():
		model = w['Model']
		ofold = int(w['OUTER_FOLD'])
		hpset = w['HPsetID']

		# extract hypers for this HPset
		hypers = _extract_hypers_for_hpset(inner_df, model, ofold, hpset)

		# set threshold from summary (mean across inner folds of the winning HPset)
		th = w[threshold_source] if threshold_source in w and pd.notna(w[threshold_source]) else None
		hypers['TH'] = float(th) if th is not None else None

		exp = {
			"ExpID": exp_id,
			"Model": model,
			"OUTER_FOLD": ofold,
			"hypers": hypers,
			"trained": False,
			"evaluated": False,
			"HPsetID": hpset,
		}
		experiments.append(exp)
		exp_id += 1

	return experiments


In [ ]:
outer_experiments = build_outer_experiments(
	summary_df=summary,
	inner_params_records=INNER_CV_parameters,  # raw records with 'hypers'
	higher_is_better=True,
	threshold_source='mean_threshold',         # uses the averaged inner-CV threshold per winner
	start_exp_id=1
)

len(outer_experiments)
save_dataset_info(outer_experiments, file="NCV_5_3_folds/OUTER_experiments.json")
